# TEMAT 13 — Detekcja jednoetapowa: YOLO (2016)

Notebook demonstracyjny. Pokazuje **ideę z artykułu z 2016** na współczesnej
implementacji ultralytics:

> Detekcja obiektów = **jeden problem regresji**. Jeden przebieg sieci
> przewiduje jednocześnie *gdzie* i *co* znajduje się na obrazie.

Sekcje:
1. Wczytanie modelu
2. Detekcja na pojedynczym obrazie
3. Kompromis `imgsz` ↔ szybkość (rdzeń tezy z 2016)
4. Ograniczenia: małe obiekty

## 1. Wczytanie modelu

Wersja `nano` — najszybsza, dobrze działa na CPU (wersja bazowa przedmiotu).

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo11n.pt')
print('Model gotowy:', model.model_name if hasattr(model, 'model_name') else 'yolo11n')

## 2. Detekcja na pojedynczym obrazie

Jeden przebieg sieci → prostokąty + klasy + pewności.

In [ ]:
results = model.predict('https://ultralytics.com/images/bus.jpg', verbose=False)
r = results[0]
print(f'Wykryto {len(r.boxes)} obiektów:')
for box in r.boxes:
    cls = model.names[int(box.cls)]
    conf = float(box.conf)
    print(f'  - {cls}: {conf:.2f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.imshow(r.plot()[..., ::-1])  # BGR -> RGB
plt.axis('off')
plt.title('YOLO — detekcja w jednym przebiegu')
plt.show()

## 3. Kompromis: rozdzielczość wejścia ↔ szybkość

Kluczowa teza artykułu z 2016: YOLO oddaje część dokładności za szybkość.
Parametr `imgsz` steruje tym kompromisem wprost.

In [ ]:
import numpy as np

dummy = np.random.randint(0, 255, (1280, 1280, 3), dtype=np.uint8)
for imgsz in [320, 640, 1280]:
    for _ in range(3):  # rozgrzewka
        model.predict(dummy, imgsz=imgsz, verbose=False)
    t = []
    for _ in range(10):
        t0 = time.perf_counter()
        model.predict(dummy, imgsz=imgsz, verbose=False)
        t.append(time.perf_counter() - t0)
    mean = np.mean(t)
    print(f'imgsz={imgsz:>4}: {mean*1000:7.1f} ms  ->  {1/mean:5.1f} FPS')

## 4. Ograniczenia — małe obiekty

Architektura jednoetapowa z 2016 miała wyraźnie niższą dokładność na
małych / zbitych obiektach (każda komórka siatki przewiduje ograniczoną
liczbę prostokątów). Porównaj detekcję na obrazie z dużymi vs. małymi
obiektami przy tym samym `imgsz`, by to zilustrować.

Podstaw własny obraz testowy w `data/` i odkomentuj poniżej.

In [ ]:
# small = model.predict('data/small_objects.jpg', imgsz=640, verbose=False)[0]
# print(f'Małe obiekty wykryte: {len(small.boxes)}')
# plt.figure(figsize=(10, 8)); plt.imshow(small.plot()[..., ::-1]); plt.axis('off'); plt.show()